# CMB Planck

Two self-contained parts:

**Part 1 - standard Planck CMB Fisher forecast** (runs directly):
CAMB with `Want_CMB` enabled, the `Planck.yaml` survey specification
($f_{\rm sky}=0.7$, $7'$ beam, $23/42\,\mu{\rm K}$-arcmin T/E noise,
$\ell\in[2,1500]$) and the Knox noise model (`legacy` also available).
The Fisher for observables `["CMB_T", "CMB_E"]` lands in `notebooks/results/`
so notebook `05-Fishers_...` can pick it up.

**Part 2 - Planck covmat vs CosmicFish noise diagnostics** (optional):
compares precomputed CosmicFish noise runs against Planck's published
covariance products.  Part 2 needs external inputs
(`uv run bash scripts/run_planck_diagnostics_suite.sh` plus the
`Planck-Results/` download); if they are absent the notebook prints how to
obtain them and every Part-2 cell is skipped gracefully instead of failing.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display

from cosmicfishpie.fishermatrix import cosmicfish
from cosmicfishpie.analysis import fisher_plot_analysis as cfa
from cosmicfishpie.analysis import fisher_matrix as fm_mod
import cosmicfishpie.analysis.fishconsumer as fco

sns.set_theme(context="talk", style="ticks")
snscolors = sns.color_palette("colorblind")


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "cosmicfishpie").exists():
            return p
    raise RuntimeError("Could not find repo root containing 'cosmicfishpie/'")


repo_root = find_repo_root(Path.cwd())
if not (repo_root / "Planck-Results").exists():
    print("NOTE: Planck-Results/ not found -> Part 2 diagnostics inputs may be missing.")
print("Repo root:", repo_root)

results_dir = Path("./results")
results_dir.mkdir(parents=True, exist_ok=True)
print(f"Results directory: {results_dir.resolve()}")

# Part 1 - Standard Planck CMB Fisher (CAMB + Planck.yaml + Knox noise)

In [ ]:
def build_camb_cmb_yaml(base_yaml_path: Path, out_yaml_path: Path) -> Path:
    """Enable CMB (TT/TE/EE) output in a copy of a CAMB boltzmann yaml."""
    cfg = yaml.safe_load(base_yaml_path.read_text(encoding="utf-8"))
    cfg.setdefault("COSMO_SETTINGS", {})
    cfg["COSMO_SETTINGS"]["WantCls"] = True
    cfg["COSMO_SETTINGS"]["Want_CMB"] = True
    cfg["COSMO_SETTINGS"]["Want_CMB_lensing"] = False
    cfg["COSMO_SETTINGS"]["Want_cl_2D_array"] = False
    out_yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return out_yaml_path


def run_fisher(name, observables, fiducial, freepars, options, specifications={}, survey_name="Euclid"):
    fm = cosmicfish.FisherMatrix(
        fiducialpars=fiducial,
        freepars=freepars,
        options=options,
        specifications=specifications,
        observables=observables,
        cosmoModel=options.get("cosmo_model", "LCDM"),
        surveyName=survey_name,
    )
    fish = fm.compute()
    print(f"Finished {name}: {fish.name}")
    return fish


def sigma_table(fisher, label, params=None):
    par_names = fisher.get_param_names()
    fid = fisher.get_param_fiducial()
    sigmas = fisher.get_confidence_bounds()

    rows = []
    for p, f0, s1 in zip(par_names, fid, sigmas):
        if params is not None and p not in params:
            continue
        pct = np.nan if f0 == 0 else 100.0 * s1 / abs(f0)
        rows.append({"fisher": label, "param": p, "fiducial": f0, "sigma_1s": s1, "percent": pct})
    return pd.DataFrame(rows)

In [ ]:
fiducial = {
    "Omegam": 0.32,
    "Omegab": 0.05,
    "h": 0.67,
    "ns": 0.96,
    "sigma8": 0.815584,
    "mnu": 0.06,
    "Neff": 3.044,
}

freepars = {
    "Omegam": 0.01,
    "Omegab": 0.01,
    "h": 0.01,
    "ns": 0.01,
    "sigma8": 0.01,
}

cosmo_params_plot = ["Omegam", "Omegab", "h", "ns", "sigma8"]

## Planck CMB Fisher (CAMB, full $\ell$ range, Knox noise)

Noise prescriptions:

* `knox` (default): paper-consistent Knox-style noise, `N_ell = w^{-1} b_ell^{-2}`
* `legacy`: historical CosmicFishPie CMB noise implementation

Observables `["CMB_T", "CMB_E"]` include the full T/E block (TT, TE, EE) over
$\ell\in$[`PLANCK_ELLMIN`, `PLANCK_ELLMAX`].  The Fisher `.txt` is exported
into `notebooks/results/` and committed from there.

In [ ]:
from importlib import resources

cfg_dir = resources.files("cosmicfishpie") / "configs"

spec_planck_path = cfg_dir / "default_survey_specifications" / "Planck.yaml"
spec_planck_base = yaml.safe_load(spec_planck_path.read_text(encoding="utf-8"))["specifications"]

# --- User-facing Planck CMB controls ---
PLANCK_NOISE_MODEL = "knox"  # "knox" (recommended) or "legacy"
PLANCK_EE_LOWELL_NOISE_BOOST = 8.0   # optional low-ell EE noise boost (Knox mode)
PLANCK_EE_LOWELL_MAX_ELL = 29
PLANCK_ELLMIN = 2
PLANCK_ELLMAX = 1500

base_camb_yaml = cfg_dir / "default_boltzmann_yaml_files" / "camb" / "default.yaml"
camb_cmb_yaml = build_camb_cmb_yaml(base_camb_yaml, results_dir / "camb_cmb_enabled.yaml")


def make_planck_spec(lmin_inclusive: int, lmax_inclusive: int):
    spec = dict(spec_planck_base)
    spec["lmin_CMB"] = int(lmin_inclusive)
    # CMB internals use half-open ell range [lmin, lmax).
    spec["lmax_CMB"] = int(lmax_inclusive) + 1
    spec["CMB_noise_model"] = PLANCK_NOISE_MODEL
    spec["CMB_EE_noise_boost_lowell"] = float(PLANCK_EE_LOWELL_NOISE_BOOST)
    spec["CMB_EE_noise_boost_lmax"] = int(PLANCK_EE_LOWELL_MAX_ELL)
    return spec


opts_planck_common = {
    "code": "camb",
    "camb_config_yaml": str(camb_cmb_yaml),
    "accuracy": 1,
    "feedback": 1,
    "derivatives": "3PT",
    "results_dir": str(results_dir) + "/",
    "cosmo_model": "LCDM",
    "survey_name": "Euclid",
    "survey_name_photo": False,
    "survey_name_spectro": False,
    "outroot": "notebook_cmb_planck_",
}

spec_planck_full = make_planck_spec(PLANCK_ELLMIN, PLANCK_ELLMAX)
opts_planck_full = {**opts_planck_common}

fish_planck = run_fisher(
    name=f"Planck CMB full ({PLANCK_NOISE_MODEL})",
    observables=["CMB_T", "CMB_E"],
    fiducial=fiducial,
    freepars=freepars,
    options=opts_planck_full,
    specifications=spec_planck_full,
    survey_name="Euclid",
)

In [ ]:
sigma_planck = sigma_table(fish_planck, f"Planck CMB [{PLANCK_NOISE_MODEL}]", cosmo_params_plot)
display(sigma_planck.set_index("param")[["sigma_1s", "percent"]])

truth = fish_planck.get_param_names_fiducial_dict()
param_labels = {p: r"$" + p + r"$" for p in cosmo_params_plot}

_ = fco.make_triangle_plot(
    fishers=[fish_planck],
    fisher_labels=[f"Planck CMB full [{PLANCK_NOISE_MODEL}]"],
    params=cosmo_params_plot,
    colors=[snscolors[0]],
    truth_values=truth,
    smooth=3,
    bins=30,
    param_labels=param_labels,
    savefile=str(results_dir / "planck_cmb_triangle.png"),
)
print("Fisher file:", fish_planck.file_name)

# Part 2 - Planck covmat vs CosmicFish noise diagnostics (optional)

Compares precomputed CosmicFish noise runs (`legacy`, `knox`, 3-part split)
against Planck's published covariance products, *without* rerunning long
Fishers.

Inputs (created by `uv run bash scripts/run_planck_diagnostics_suite.sh`
plus the `Planck-Results/` download from the
[Planck legacy archive](https://wiki.cosmos.esa.int/planck-legacy-archive/index.php/Cosmological_Parameters)):
`tmp/planck_bestfit_theta*/...` Fisher files and a Planck `.covmat`/`.likestats`
pair.  If any input is missing, the next cell only *reports* what is missing
and the Part-2 cells below skip cleanly (Part 1 is unaffected).

## Load precomputed Fishers and Planck references

In [ ]:
paths = {
    "legacy_full": repo_root / "tmp/planck_bestfit_theta/CosmicFish_v1.3.0_planck_bestfit_theta_primary__CMB_TCMB_E_FM.txt",
    "knox_full": repo_root / "tmp/planck_bestfit_theta_knox_full/CosmicFish_v1.3.0_planck_bestfit_theta_primary__CMB_TCMB_E_FM.txt",
    "knox_full_l1500": repo_root / "tmp/planck_bestfit_theta_knox_full_l1500/CosmicFish_v1.3.0_planck_bestfit_theta_primary__CMB_TCMB_E_FM.txt",
    "knox_high": repo_root / "tmp/planck_bestfit_theta_knox_3part/high_te/CosmicFish_v1.3.0_planck_bestfit_theta_primary__CMB_TCMB_E_FM.txt",
    "knox_low_t": repo_root / "tmp/planck_bestfit_theta_knox_3part/low_t/CosmicFish_v1.3.0_planck_bestfit_theta_primary__CMB_T_FM.txt",
    "knox_low_e": repo_root / "tmp/planck_bestfit_theta_knox_3part/low_e/CosmicFish_v1.3.0_planck_bestfit_theta_primary__CMB_E_FM.txt",
    "planck_dir": repo_root / "Planck-Results/COM_CosmoParams_base-plikHM-TTTEEE-lowl-lowE_R3.00/base/plikHM_TTTEEE_lowl_lowE",
}

for key, val in paths.items():
    print(f"{key:>12}: {val}")

required = ["legacy_full", "knox_full", "knox_full_l1500", "knox_high", "knox_low_t", "knox_low_e", "planck_dir"]
missing = [k for k in required if not paths[k].exists()]
DIAGNOSTICS_AVAILABLE = not missing
if missing:
    print("Part 2 diagnostics inputs missing:", ", ".join(missing))
    print("Run:  uv run bash scripts/run_planck_diagnostics_suite.sh")
    print("and download Planck-Results/ (see markdown above). Part-2 cells are skipped below.")
else:
    print("All Part-2 inputs found.")

In [ ]:
if not DIAGNOSTICS_AVAILABLE:
    print("skip (Part-2 inputs missing)")
else:
    f_legacy = fm_mod.fisher_matrix(file_name=str(paths["legacy_full"]))
    f_knox_full = fm_mod.fisher_matrix(file_name=str(paths["knox_full"]))
    f_knox_full_l1500 = fm_mod.fisher_matrix(file_name=str(paths["knox_full_l1500"]))
    f_knox_high = fm_mod.fisher_matrix(file_name=str(paths["knox_high"]))
    f_knox_low_t = fm_mod.fisher_matrix(file_name=str(paths["knox_low_t"]))
    f_knox_low_e = fm_mod.fisher_matrix(file_name=str(paths["knox_low_e"]))

    f_knox_3part = f_knox_high + f_knox_low_t + f_knox_low_e
    f_knox_3part.name = "CosmicFish_v1.3.0_planck_bestfit_theta_knox_3part_FM"

    out_threepart = repo_root / "tmp/planck_bestfit_theta_knox_3part/combined"
    out_threepart.mkdir(parents=True, exist_ok=True)
    out_threepart_root = out_threepart / f_knox_3part.name
    f_knox_3part.save_to_file(str(out_threepart_root))

    print("Saved combined 3-part Fisher:", str(out_threepart_root) + ".txt")

In [17]:
def parse_likestats_bestfit(path: Path) -> dict[str, float]:
    out = {}
    in_table = False
    for line in path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s:
            continue
        if s.startswith("parameter"):
            in_table = True
            continue
        if not in_table:
            continue
        parts = s.split()
        if len(parts) < 2:
            continue
        try:
            out[parts[0]] = float(parts[1])
        except ValueError:
            continue
    return out


def read_covmat(path: Path) -> tuple[list[str], np.ndarray]:
    lines = [ln for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    header = lines[0].strip()
    names = header.lstrip("#").strip().split()
    cov = np.loadtxt(path)
    cov = np.atleast_2d(cov).astype(float)
    return names, cov


def planck_covmat_as_fishers(planck_dir: Path, chain_root: str, params_cfp: list[str]):
    cov_path = planck_dir / "dist" / f"{chain_root}.covmat"
    like_path = planck_dir / "dist" / f"{chain_root}.likestats"

    cov_names, cov_full = read_covmat(cov_path)
    best = parse_likestats_bestfit(like_path)

    cfp_to_planck = {"ombh2": "omegabh2", "omch2": "omegach2", "logAs": "logA"}
    planck_names = [cfp_to_planck.get(p, p) for p in params_cfp]
    idx = [cov_names.index(p) for p in planck_names]

    cov_sub = cov_full[np.ix_(idx, idx)]
    fisher_marg = np.linalg.pinv(cov_sub)

    fisher_full = np.linalg.pinv(cov_full)
    fisher_cond = fisher_full[np.ix_(idx, idx)]

    fid = np.array([best.get(pn, 0.0) for pn in planck_names], dtype=float)
    latex = [r"\Omega_b h^2", r"\Omega_c h^2", r"100\theta_{\rm MC}", r"\tau", r"\ln(10^{10}A_s)", r"n_s"]

    f_marg = fm_mod.fisher_matrix(
        fisher_matrix=fisher_marg,
        param_names=params_cfp,
        param_names_latex=latex,
        fiducial=fid,
        name="Planck_covmat_marginalized",
    )
    f_cond = fm_mod.fisher_matrix(
        fisher_matrix=fisher_cond,
        param_names=params_cfp,
        param_names_latex=latex,
        fiducial=fid,
        name="Planck_covmat_conditional",
    )
    return f_marg, f_cond


f_cov_marg, f_cov_cond = planck_covmat_as_fishers(
    paths["planck_dir"], "base_plikHM_TTTEEE_lowl_lowE", params
)

cov_out = repo_root / "tmp/planck_bestfit_theta_knox_full/covmat_compare"
cov_out.mkdir(parents=True, exist_ok=True)
f_cov_marg.save_to_file(str(cov_out / "Planck_covmat_Gaussian_marginalized"))
f_cov_cond.save_to_file(str(cov_out / "Planck_covmat_Gaussian_conditional"))

print("Saved:", cov_out / "Planck_covmat_Gaussian_marginalized.txt")
print("Saved:", cov_out / "Planck_covmat_Gaussian_conditional.txt")

Saved: /home/casas/Cosmo/cosmicfishpie/tmp/planck_bestfit_theta_knox_full/covmat_compare/Planck_covmat_Gaussian_marginalized.txt
Saved: /home/casas/Cosmo/cosmicfishpie/tmp/planck_bestfit_theta_knox_full/covmat_compare/Planck_covmat_Gaussian_conditional.txt


## Sigma Comparison Table

In [ ]:
if not DIAGNOSTICS_AVAILABLE:
    print("skip (Part-2 inputs missing)")
else:
    def sigma_map(fisher):
        names = [str(x) for x in fisher.get_param_names()]
        sig = [float(x) for x in fisher.get_confidence_bounds()]
        return dict(zip(names, sig))

    s_cov_marg = sigma_map(f_cov_marg)
    s_cov_cond = sigma_map(f_cov_cond)
    s_legacy = sigma_map(f_legacy)
    s_knox = sigma_map(f_knox_full)
    s_knox_l1500 = sigma_map(f_knox_full_l1500)
    s_3p = sigma_map(f_knox_3part)

    rows = []
    for p in params:
        rows.append(
            {
                "parameter": p,
                "sigma_cov_marg": s_cov_marg[p],
                "sigma_cov_cond": s_cov_cond[p],
                "sigma_legacy": s_legacy[p],
                "sigma_knox_full": s_knox[p],
                "sigma_knox_full_l1500": s_knox_l1500[p],
                "sigma_knox_3part": s_3p[p],
                "legacy/cov_marg": s_legacy[p] / s_cov_marg[p],
                "knox_full/cov_marg": s_knox[p] / s_cov_marg[p],
                "knox_3part/cov_marg": s_3p[p] / s_cov_marg[p],
                "legacy/cov_cond": s_legacy[p] / s_cov_cond[p],
                "knox_full/cov_cond": s_knox[p] / s_cov_cond[p],
                "knox_full_l1500/cov_cond": s_knox_l1500[p] / s_cov_cond[p],
                "knox_3part/cov_cond": s_3p[p] / s_cov_cond[p],
            }
        )

    df_sigma = pd.DataFrame(rows)
    display(df_sigma)

## Correlation Metrics

In [ ]:
if not DIAGNOSTICS_AVAILABLE:
    print("skip (Part-2 inputs missing)")
else:
    def corr_matrix(fisher, pnames):
        cov = np.asarray(fisher.inverse_fisher_matrix(), dtype=float)
        idx = [fisher.get_param_index(p) for p in pnames]
        sub = cov[np.ix_(idx, idx)]
        d = np.sqrt(np.diag(sub))
        return sub / np.outer(d, d)


    def corr_metrics(fisher, ref, pnames):
        a = corr_matrix(fisher, pnames)
        b = corr_matrix(ref, pnames)
        iu = np.triu_indices(len(pnames), 1)
        da = a[iu]
        db = b[iu]
        diff = da - db
        return {
            "sign_match_fraction": float(np.mean(np.sign(da) == np.sign(db))),
            "mean_abs_delta_corr": float(np.mean(np.abs(diff))),
            "rms_delta_corr": float(np.sqrt(np.mean(diff * diff))),
            "max_abs_delta_corr": float(np.max(np.abs(diff))),
        }

    models = {
        "legacy_full": f_legacy,
        "knox_full": f_knox_full,
        "knox_full_l1500": f_knox_full_l1500,
        "knox_3part": f_knox_3part,
    }
    refs = {
        "cov_marg": f_cov_marg,
        "cov_cond": f_cov_cond,
    }

    corr_rows = []
    for model_name, model_f in models.items():
        for ref_name, ref_f in refs.items():
            corr_rows.append({"fisher": model_name, "reference": ref_name, **corr_metrics(model_f, ref_f, params)})

    df_corr = pd.DataFrame(corr_rows)
    display(df_corr)

## Triangle Plot

In [ ]:
if not DIAGNOSTICS_AVAILABLE:
    print("skip (Part-2 inputs missing)")
else:
    triangle_path = repo_root / "tmp/planck_bestfit_theta_knox_full/covmat_compare/triangle_legacy_knox_3part_covmarg_covcond.png"
    triangle_path.parent.mkdir(parents=True, exist_ok=True)

    fco.make_triangle_plot(
        #fishers=[f_legacy, f_knox_full, f_knox_3part, f_cov_marg, f_cov_cond],
        fishers=[f_knox_full, f_knox_full_l1500, f_cov_cond],
        #fisher_labels=["CFP legacy", "CFP knox full", "CFP knox 3-part", "Planck cov marg", "Planck cov cond"],
        fisher_labels=["CFP knox full", "CFP knox full l1500", "Planck cov cond"],
        params=params,
        #colors=["#7f7f7f", "#1f77b4", "#2ca02c", "#d62728", "#9467bd"],
        colors=["#1f77b4", "#2ca02c", "#d62728", "#9467bd"],
        shade_fisher=False,
        ls_fisher=["-", "--", "-.", ":", "-."],
        lw_fisher=[2.0, 2.0, 2.0, 2.0, 2.0],
        savefile=str(triangle_path),
    )

    print("Saved:", triangle_path)

## High/Low Information Split (Diagonal Fisher Entries)

In [ ]:
if not DIAGNOSTICS_AVAILABLE:
    print("skip (Part-2 inputs missing)")
else:
    F_high = np.asarray(f_knox_high.get_fisher_matrix(), dtype=float)
    F_low_t = np.asarray(f_knox_low_t.get_fisher_matrix(), dtype=float)
    F_low_e = np.asarray(f_knox_low_e.get_fisher_matrix(), dtype=float)
    F_tot = np.asarray(f_knox_3part.get_fisher_matrix(), dtype=float)

    idx = {p: f_knox_3part.get_param_index(p) for p in params}
    diag_rows = []
    for p in params:
        i = idx[p]
        d_h = float(F_high[i, i])
        d_t = float(F_low_t[i, i])
        d_e = float(F_low_e[i, i])
        d_tot = float(F_tot[i, i])
        diag_rows.append(
            {
                "parameter": p,
                "diag_high": d_h,
                "diag_low_t": d_t,
                "diag_low_e": d_e,
                "diag_total": d_tot,
                "frac_high": d_h / d_tot if d_tot != 0 else np.nan,
                "frac_low_t": d_t / d_tot if d_tot != 0 else np.nan,
                "frac_low_e": d_e / d_tot if d_tot != 0 else np.nan,
            }
        )

    df_diag = pd.DataFrame(diag_rows)
    display(df_diag)

    x = np.arange(len(params))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x, df_diag["frac_high"], label="high TE/TT/EE", alpha=0.85)
    ax.bar(x, df_diag["frac_low_t"], bottom=df_diag["frac_high"], label="low TT", alpha=0.85)
    ax.bar(
        x,
        df_diag["frac_low_e"],
        bottom=df_diag["frac_high"] + df_diag["frac_low_t"],
        label="low EE",
        alpha=0.85,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(params)
    ax.set_ylabel("Fraction of diagonal Fisher info")
    ax.set_title("Knox 3-part decomposition (diagonal entries)")
    ax.legend()
    ax.grid(alpha=0.2)
    plt.tight_layout()